In [1]:
import os
import warnings

# Нужен для загрузки данных и скриптов
os.chdir("/Data/EEG-Visual-Experiment")
warnings.filterwarnings('ignore')

#### Подгрузка пакетов, датасетов и таблицы отношений

In [10]:
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt

from typing import *
from Scripts.Data_Loader import EIRDataset

In [3]:
path_perception = "./Generated/Data_Pattern/"
path_imagery = "./Generated/Data_Train/"

ds_perception = EIRDataset(path_perception, task_type="geometric", n_jobs=70)  # Do not set the `n_jobs` = 72 (Can crash the server)
ds_imagery = EIRDataset(path_imagery, task_type="geometric", n_jobs=70)  # Do not set the `n_jobs` = 72 (Can crash the server)

Loading .fif files: 100%|██████████| 840/840 [00:35<00:00, 23.96it/s]


In [4]:
def create_perception_imagery_table(
    perception: EIRDataset,
    imagery: EIRDataset
) -> pd.DataFrame:
    ref_perception = pd.DataFrame([
        {
            "pdsi": idx,
            "subject_id": meta["subject_id"],
            "trial_id": meta["trial_id"],
            "pattern_id": pattern_id,
        }
        for idx, (_, _, meta, pattern_id, _) in enumerate(perception)
    ])
    ref_imagery = pd.DataFrame([
        {
            "idsi": idx,
            "subject_id": meta["subject_id"],
            "trial_id": meta["trial_id"],
            "pattern_id": pattern_id,
        }
        for idx, (_, _, meta, pattern_id, _) in enumerate(ds_imagery)
    ])
    
    keys = ["subject_id", "trial_id", "pattern_id"]

    ref_grouped = (
        ref_perception
        .merge(
            ref_imagery.groupby(keys, as_index=False)["idsi"].agg(list),
            on=keys,
            how="left",
        )
        .rename(columns={
            "pdsi": "perception_dataset_index",
            "idsi": "imagery_dataset_indexes"
        })
    )
    
    ref_grouped["imagery_dataset_indexes"] = ref_grouped["imagery_dataset_indexes"].apply(lambda x: x if isinstance(x, list) else [])

    return ref_grouped.copy()

In [5]:
df = create_perception_imagery_table(ds_perception, ds_imagery)
df.head()

,perception_dataset_index,subject_id,trial_id,pattern_id,imagery_dataset_indexes
0,0,4,1,6,"[182, 185, 186, 187]"
1,1,4,1,7,"[184, 188]"
2,2,4,1,3,[183]
3,3,4,1,12,"[189, 192, 193, 194]"
4,4,4,1,8,"[191, 195]"


#### Сравнение топограм для одного объекта, одного паттерна

In [16]:
def montage_electrode_schema(
    raw: mne.io.Raw,
    path: Optional[None] = None
) -> mne.io.Raw:
    s_raw = raw.copy().pick_types(eeg=True)
    if path is not None:
        montage = mne.channels.read_custom_montage(path)
        s_raw.set_montage(montage, on_missing="warn", match_case=False)
    return s_raw

In [25]:
def get_data(
    data_frame: pd.DataFrame,
    perception: EIRDataset,
    imagery: EIRDataset,
    *,
    subject_id: int,
    trial_id: int,
    pattern_id: int,
    path: Optional[None] = None
) -> Tuple[mne.io.Raw, List[mne.io.Raw]]:
    rows = data_frame[(data_frame["subject_id"] == subject_id) & (data_frame["trial_id"] == trial_id) & (data_frame["pattern_id"] == pattern_id)]

    if len(rows) != 1:
        raise ValueError(f"By parameters {subject_id=} {trial_id=} {pattern_id=} found {len(rows)=} objects. Should to be 1")
    
    perception_dataset_index = rows["perception_dataset_index"].iloc[0]
    imagery_dataset_indexes = rows["imagery_dataset_indexes"].iloc[0]

    perception_eeg, *_= perception[perception_dataset_index]
    r_perception_eeg = montage_electrode_schema(perception_eeg, path).copy()

    imagery_eegs = []
    for imagery_dataset_index in imagery_dataset_indexes:
        imagery_eeg, *_= imagery[imagery_dataset_index]
        r_imagery_eeg = montage_electrode_schema(imagery_eeg, path).copy()
        imagery_eegs.append(r_imagery_eeg)

    return r_perception_eeg, imagery_eegs

In [27]:
get_data(
    df,
    ds_perception,
    ds_imagery,
    subject_id=1,
    trial_id=1,
    pattern_id=10,
    path="./Supplementary/CMA-64_REF.bvef"
)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


(<Raw | patt_EEG_1.fif, 63 x 26001 (26.0 s), ~12.6 MB, data loaded>,
 [<Raw | exec_EEG_2.fif, 63 x 16001 (16.0 s), ~7.8 MB, data loaded>,
  <Raw | exec_EEG_3.fif, 63 x 16001 (16.0 s), ~7.8 MB, data loaded>,
  <Raw | exec_EEG_5.fif, 63 x 16001 (16.0 s), ~7.8 MB, data loaded>,
  <Raw | exec_EEG_7.fif, 63 x 16001 (16.0 s), ~7.8 MB, data loaded>])